# BLIP-2 captioning — pipeline stage 2

**What this notebook shows.** How the BLIP-2 captioner produces the short product captions that are fused with CLIP image embeddings to form the gallery vectors in conditions B and C.

BLIP-2 stays **frozen** in our pipeline — we never fine-tune it. The captions are generated once, offline, and serialised to a `captions.json` file keyed by image path.

**Model.** `Salesforce/blip2-opt-2.7b` loaded in 8-bit (`bitsandbytes`) — ~6 GB VRAM. Falls back to fp32 on CPU if no GPU.

**Usage.** Iterate the entire train + gallery split (~38k images) and save `captions.json`. For a 14k T4 it takes about 4–5 h end-to-end. On Kaggle, run with **GPU enabled + Internet enabled** (HF model download).

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import json
import os
from pathlib import Path

import torch
from PIL import Image
from tqdm import tqdm
from transformers import Blip2ForConditionalGeneration, Blip2Processor, BitsAndBytesConfig

# Path setup — works on Kaggle (where /kaggle/input is mounted) and locally.
if Path('/kaggle/input').exists():
    DATASET_ROOT = Path('/kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset')
    OUTPUT_DIR   = Path('/kaggle/working')
else:
    DATASET_ROOT = Path('vr_final_proj_dataset')
    OUTPUT_DIR   = Path('artifacts'); OUTPUT_DIR.mkdir(exist_ok=True)

IMG_ROOT   = DATASET_ROOT / 'img' / 'img'
BBOX_FILE  = DATASET_ROOT / 'Anno' / 'list_bbox_inshop.txt'
SPLIT_FILE = DATASET_ROOT / 'eval' / 'list_eval_partition.txt'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
print('IMG_ROOT exists  :', IMG_ROOT.exists())
print('SPLIT_FILE exists:', SPLIT_FILE.exists())

In [ ]:
# ── 2. Load BLIP-2 OPT-2.7b in 8-bit (≈ 6 GB VRAM) ────────────────────────
MODEL = 'Salesforce/blip2-opt-2.7b'
processor = Blip2Processor.from_pretrained(MODEL)

if DEVICE == 'cuda':
    bnb = BitsAndBytesConfig(load_in_8bit=True)
    model = Blip2ForConditionalGeneration.from_pretrained(
        MODEL, quantization_config=bnb, device_map='auto'
    )
else:
    model = Blip2ForConditionalGeneration.from_pretrained(MODEL)
model.eval()
print('BLIP-2 ready.')

In [ ]:
# ── 3. Helpers: bbox crop + caption-one-image ─────────────────────────────
def parse_bbox_file(path):
    bboxes = {}
    with open(path) as f: lines = f.readlines()
    for line in lines[2:]:
        parts = line.strip().split()
        if len(parts) < 7: continue
        raw = parts[0]
        img_name = raw[len('img/'):] if raw.startswith('img/') else raw
        bboxes[img_name] = (int(parts[3]), int(parts[4]), int(parts[5]), int(parts[6]))
    return bboxes


def parse_split_file(path):
    rows = []
    with open(path) as f: lines = f.readlines()
    for line in lines[2:]:
        parts = line.strip().split()
        if len(parts) < 3: continue
        raw = parts[0]
        img_name = raw[len('img/'):] if raw.startswith('img/') else raw
        rows.append({'image_name': img_name, 'item_id': parts[1], 'split': parts[2]})
    return rows


def bbox_crop(pil, bbox, pad=0.05):
    W, H = pil.size
    x1, y1, x2, y2 = bbox
    px = int((x2-x1)*pad); py = int((y2-y1)*pad)
    x1=max(0,x1-px); y1=max(0,y1-py); x2=min(W,x2+px); y2=min(H,y2+py)
    return pil.crop((x1,y1,x2,y2)) if x2>x1 and y2>y1 else pil


@torch.no_grad()
def caption_image(pil_image, max_new_tokens=12, num_beams=3):
    inputs = processor(images=pil_image, return_tensors='pt').to(DEVICE)
    if DEVICE == 'cuda':
        inputs = {k: (v.half() if v.dtype.is_floating_point else v) for k, v in inputs.items()}
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=num_beams,
                         no_repeat_ngram_size=2)
    return processor.batch_decode(out, skip_special_tokens=True)[0].strip()

In [ ]:
# ── 4. Demo on 5 sample gallery images ────────────────────────────────────
import matplotlib.pyplot as plt

rows = parse_split_file(SPLIT_FILE)
bbox_map = parse_bbox_file(BBOX_FILE)
gallery = [r for r in rows if r['split'] == 'gallery'][:5]

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, r in zip(axes, gallery):
    pil = Image.open(IMG_ROOT / r['image_name']).convert('RGB')
    crop = bbox_crop(pil, bbox_map.get(r['image_name']))
    cap = caption_image(crop)
    ax.imshow(crop); ax.axis('off'); ax.set_title(f"{r['item_id']}\n{cap}", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# ── 5. Full pass over train + gallery (~38k images, 4-5 h on T4) ──────────
# Skip this cell unless you actually want to (re-)generate captions.json.
RUN_FULL_PASS = False  # toggle to True to actually run

if RUN_FULL_PASS:
    targets = [r for r in rows if r['split'] in ('train', 'gallery')]
    out_path = OUTPUT_DIR / 'captions.json'
    captions = json.load(open(out_path)) if out_path.exists() else {}
    print(f'Resuming from {len(captions)} existing captions; {len(targets)} total.')

    pbar = tqdm(targets)
    for i, r in enumerate(pbar):
        if r['image_name'] in captions:
            continue
        try:
            pil = Image.open(IMG_ROOT / r['image_name']).convert('RGB')
            crop = bbox_crop(pil, bbox_map.get(r['image_name']))
            captions[r['image_name']] = caption_image(crop)
        except Exception as e:
            captions[r['image_name']] = ''  # leave blank, doesn't break α<1 fusion
        if (i + 1) % 1000 == 0:
            json.dump(captions, open(out_path, 'w'))
            pbar.set_postfix({'saved': len(captions)})
    json.dump(captions, open(out_path, 'w'))
    print(f'Saved {len(captions)} captions → {out_path}')
else:
    print('RUN_FULL_PASS=False — skipping the 4-5 h pass. Toggle the flag to actually run.')

### What to take away from this notebook

1. **Captions are short** — typically 2–5 words like *"black floral print mini dress"*. This is by design: BLIP-2 is asked to describe a single garment photo, and the OPT-2.7b language head with `num_beams=3` favours concise outputs.
2. **The text vocabulary is small** — there are ~38k captions but only a few hundred distinct adjective+noun combinations. This limited diversity is why BLIP-2 ITM re-ranking doesn't help on this dataset (see `blip2-itm.ipynb` and `report.md` §6.2 finding #5): top-K candidates share near-identical captions.
3. **No fine-tuning needed.** BLIP-2 captions are good enough to add ~2 pp Recall@10 over vision-only when fused with α=0.7 (condition A → B in the report).